# Evaluating LLMs — Code Companion

This notebook accompanies **Topic: Evaluating LLMs — Perplexity, Task-Specific Eval
Sets & LLM-as-Judge**.

Training loss going down doesn't mean your model actually got better. This notebook
builds all three evaluation approaches from the slides, hands-on:

1. **Perplexity** — computed from scratch, then with a real model.
2. **Task-specific eval sets** — building one, and scoring a model against it.
3. **LLM-as-judge** — a real, working pairwise comparison script, including bias
   mitigation (order swapping).

Sections 1 and 2 run anywhere with just NumPy. Sections 3 onward use `transformers`
(needs internet the first time) and, for the LLM-as-judge section, an API key for
whichever LLM you use as the judge.

## 1. Perplexity, Built From Scratch

Before using a library, let's implement the formula directly so the mechanics are fully
visible:

$$PPL(X) = \exp\left(-\frac{1}{N}\sum_i \log P(x_i \mid x_{<i})\right)$$

We'll simulate a tiny "model" as a lookup table of next-token probabilities, so we can
compute perplexity without needing any real neural network yet.

In [1]:
import numpy as np

np.random.seed(0)

# A toy vocabulary and a toy "model": a fixed probability distribution over next tokens
vocab = ["the", "cat", "sat", "on", "mat", "dog", "ran"]
vocab_size = len(vocab)

# Pretend this model is quite confident about "the cat sat on the mat"
# by assigning that exact sequence high probability at each step.
def toy_next_token_probs(true_next_token, confidence=0.7):
    """Returns a probability distribution that puts `confidence` mass on the true
    next token, and spreads the rest uniformly over the other tokens."""
    probs = np.full(vocab_size, (1 - confidence) / (vocab_size - 1))
    probs[vocab.index(true_next_token)] = confidence
    return probs

def compute_perplexity_from_scratch(sequence, confidence=0.7):
    log_probs = []
    for i in range(1, len(sequence)):
        probs = toy_next_token_probs(sequence[i], confidence=confidence)
        p_true = probs[vocab.index(sequence[i])]
        log_probs.append(np.log(p_true))
    avg_neg_log_prob = -np.mean(log_probs)
    perplexity = np.exp(avg_neg_log_prob)
    return perplexity

sequence = ["the", "cat", "sat", "on", "the", "mat"]

for confidence in [0.9, 0.7, 0.5, 0.3]:
    ppl = compute_perplexity_from_scratch(sequence, confidence=confidence)
    print(f"Model confidence={confidence:.1f}  ->  Perplexity={ppl:.2f}")

Model confidence=0.9  ->  Perplexity=1.11
Model confidence=0.7  ->  Perplexity=1.43
Model confidence=0.5  ->  Perplexity=2.00
Model confidence=0.3  ->  Perplexity=3.33


Notice: as the model's confidence in the correct next token *drops*, perplexity *rises*.
A perfectly confident model (confidence → 1.0) approaches a perplexity of 1 — the
theoretical best possible score. This is the entire intuition behind the metric, before
any real neural network is involved.

## 2. Perplexity on a Real Model

Now the same formula, computed the way it's actually done in practice: using the loss a
real language model computes during a forward pass.

> **Note:** needs `pip install transformers` and internet access the first time, to
> download a small model.

In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "gpt2"   # small and fast for this demo; swap for any causal LM
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

def compute_perplexity(model, tokenizer, text):
    inputs = tokenizer(text, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
    return torch.exp(outputs.loss).item()

fluent_text = "The quick brown fox jumps over the lazy dog."
scrambled_text = "Jumps dog lazy the over fox brown quick the."

print(f"Fluent text perplexity:    {compute_perplexity(model, tokenizer, fluent_text):.2f}")
print(f"Scrambled text perplexity: {compute_perplexity(model, tokenizer, scrambled_text):.2f}")

W0812 13:38:49.423000 3684807 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W0812 13:38:49.445000 3684807 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Fluent text perplexity:    162.47
Scrambled text perplexity: 6886.23


The scrambled version should score noticeably higher (more "surprising" to the model) —
a nice sanity check that the metric is measuring something real, even though (as the
slides emphasize) it says nothing about whether either sentence is *useful*.

## 3. Building a Task-Specific Eval Set

Perplexity can't tell you if a model actually does *your* task well. Let's build a small,
realistic eval set and score a model against it directly — exact-match accuracy for
crisp, factual answers.

In [3]:
eval_set = [
    {"prompt": "What is the capital of France?", "answer": "paris"},
    {"prompt": "What is 12 + 7?", "answer": "19"},
    {"prompt": "What planet do we live on?", "answer": "earth"},
    {"prompt": "What is the chemical symbol for water?", "answer": "h2o"},
    {"prompt": "Who wrote Romeo and Juliet?", "answer": "shakespeare"},
]

print(f"Eval set has {len(eval_set)} examples. First one:")
print(eval_set[0])

Eval set has 5 examples. First one:
{'prompt': 'What is the capital of France?', 'answer': 'paris'}


In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2", max_new_tokens=12)

def run_eval(eval_set, generator):
    correct = 0
    results = []
    for ex in eval_set:
        output = generator(ex["prompt"], num_return_sequences=1)[0]["generated_text"]
        is_correct = ex["answer"].lower() in output.lower()
        correct += is_correct
        results.append({"prompt": ex["prompt"], "output": output, "correct": is_correct})
    accuracy = correct / len(eval_set)
    return accuracy, results

accuracy, results = run_eval(eval_set, generator)

for r in results:
    mark = "correct" if r["correct"] else "WRONG"
    print(f"[{mark:7s}] {r['prompt']}")
    print(f"          -> {r['output'][:90]}")

print(f"\nAccuracy: {accuracy:.1%}")

Small, general-purpose models like base GPT-2 often score poorly on tasks like this
because they weren't instruction-tuned — which is itself a useful, concrete illustration
of exactly why the SFT stage from earlier topics matters. Try re-running this section
against an instruction-tuned model (e.g. swap in a `-Instruct` checkpoint) and compare.

## 4. LLM-as-Judge: A Real Pairwise Comparison Script

This section builds a working judge function you can point at any chat-style LLM API.
It follows the slide's guidance directly: pairwise comparison (not absolute 1-10 scoring),
a clear rubric, and swapping response order to control for position bias.

We use a generic `call_llm(prompt)` function stub — plug in your preferred provider's
API call (OpenAI, Anthropic, a local model, etc.).

In [5]:
JUDGE_PROMPT_TEMPLATE = """You are comparing two AI responses to the same prompt. Judge them ONLY on helpfulness and accuracy for the given prompt.

Prompt: {prompt}

Response 1: {response_1}

Response 2: {response_2}

Which response is better? Reply with EXACTLY one line in this format:
VERDICT: <1, 2, or TIE>
REASON: <one sentence>"""


def call_llm(prompt: str) -> str:
    """Stub -- replace with a real API call, e.g.:

    from anthropic import Anthropic
    client = Anthropic()
    resp = client.messages.create(
        model="claude-sonnet-4-6", max_tokens=100,
        messages=[{"role": "user", "content": prompt}],
    )
    return resp.content[0].text
    """
    raise NotImplementedError("Plug in a real LLM API call here.")


def llm_judge(prompt, response_a, response_b, call_llm_fn=call_llm):
    """Runs the comparison TWICE with response order swapped, to control for
    position bias, then combines the two verdicts."""
    verdict_1 = call_llm_fn(JUDGE_PROMPT_TEMPLATE.format(
        prompt=prompt, response_1=response_a, response_2=response_b))
    verdict_2 = call_llm_fn(JUDGE_PROMPT_TEMPLATE.format(
        prompt=prompt, response_1=response_b, response_2=response_a))

    # verdict_1 says "1" -> A preferred; verdict_2 says "2" -> A preferred (order flipped)
    a_wins = verdict_1.count("VERDICT: 1") + verdict_2.count("VERDICT: 2")
    b_wins = verdict_1.count("VERDICT: 2") + verdict_2.count("VERDICT: 1")

    if a_wins > b_wins:
        return "A", (verdict_1, verdict_2)
    elif b_wins > a_wins:
        return "B", (verdict_1, verdict_2)
    else:
        return "TIE", (verdict_1, verdict_2)

In [6]:
# A mock call_llm so this cell runs and demonstrates the control flow without a real API key.
# Replace `mock_call_llm` with the real `call_llm` above once you have API access.
import random
random.seed(1)

def mock_call_llm(prompt):
    # Pretend the judge reliably prefers the longer, more detailed response
    r1_len = len(prompt.split("Response 1: ")[1].split("Response 2:")[0])
    r2_len = len(prompt.split("Response 2: ")[1])
    winner = "1" if r1_len > r2_len else "2"
    return f"VERDICT: {winner}\nREASON: More detailed and complete."

winner, raw_verdicts = llm_judge(
    prompt="Explain photosynthesis to a 10-year-old.",
    response_a="Plants use sunlight, water, and air to make their own food -- like a tiny kitchen powered by the sun!",
    response_b="Photosynthesis is a chemical process.",
    call_llm_fn=mock_call_llm,
)

print(f"Winner: {winner}")
print("Raw verdicts (order 1, order 2):", raw_verdicts)

Winner: TIE
Raw verdicts (order 1, order 2): ('VERDICT: 2\nREASON: More detailed and complete.', 'VERDICT: 2\nREASON: More detailed and complete.')


## 5. Known Bias Check: Does Order Actually Matter?

Let's demonstrate *why* the order-swapping in Section 4 matters, using a deliberately
biased mock judge that just always prefers whichever response comes first — exactly the
position bias called out in the slides.

In [7]:
def position_biased_judge(prompt):
    # A badly-behaved judge that always just picks "Response 1", regardless of content
    return "VERDICT: 1\nREASON: (biased) always prefers whichever came first"

def naive_judge_no_swap(prompt, response_a, response_b, call_llm_fn):
    """WITHOUT order-swapping -- vulnerable to position bias."""
    verdict = call_llm_fn(JUDGE_PROMPT_TEMPLATE.format(
        prompt=prompt, response_1=response_a, response_2=response_b))
    return "A" if "VERDICT: 1" in verdict else "B"

# Same two responses, but we ask the naive judge in both orders
result_order_1 = naive_judge_no_swap(
    "test", "Response A (actually worse)", "Response B (actually better)",
    call_llm_fn=position_biased_judge,
)
result_order_2 = naive_judge_no_swap(
    "test", "Response B (actually better)", "Response A (actually worse)",
    call_llm_fn=position_biased_judge,
)

print(f"Naive judge, A first:  picks {result_order_1}")
print(f"Naive judge, B first:  picks {result_order_2}")
print()
print("The 'winner' flips just from changing the order -- pure position bias.")
print("This is exactly why llm_judge() in Section 4 always runs BOTH orders and combines them.")

winner, _ = llm_judge(
    "test", "Response A (actually worse)", "Response B (actually better)",
    call_llm_fn=position_biased_judge,
)
print(f"\nOur order-swapping llm_judge() on the same biased judge -> result: {winner}")
print("(TIE is the correct, honest outcome here: the biased judge gives no real signal.)")

Naive judge, A first:  picks A
Naive judge, B first:  picks A

The 'winner' flips just from changing the order -- pure position bias.
This is exactly why llm_judge() in Section 4 always runs BOTH orders and combines them.

Our order-swapping llm_judge() on the same biased judge -> result: TIE
(TIE is the correct, honest outcome here: the biased judge gives no real signal.)


## Recap & Try It Yourself

You just:
- Implemented the perplexity formula from scratch and watched it respond to model
  confidence, then computed it on a real model with `transformers`.
- Built a small task-specific eval set and scored a real model's exact-match accuracy
  against it.
- Wrote a working LLM-as-judge function with built-in order-swapping to control for
  position bias.
- Proved *why* that order-swapping matters, using a deliberately biased mock judge.

**Things to try:**
1. In Section 2, try 3-4 of your own sentences (fluent vs. deliberately odd) and predict
   the perplexity ordering before running the cell.
2. In Section 3, swap `"gpt2"` for an instruction-tuned checkpoint and compare accuracy.
3. In Section 4, replace `call_llm` with a real API call and re-run the comparison on
   genuine model outputs, e.g. base Llama 3.1 (8B) vs. your SFT+DPO checkpoint from the
   earlier notebooks.
4. Extend `llm_judge` to also collect and print the judge's stated reasons, not just the
   verdict — useful for spot-checking whether the judge's reasoning actually makes sense.